# 🚀 Kaggle Dual T4 GPU AI Model Runner
- **Ускоритель**: 2x NVIDIA Tesla T4 (32 ГБ суммарной VRAM).
- **Модель**: Qwen/Qwen2.5-7B-Instruct (мультиязычная модель с балансировкой по двум GPU).


In [ ]:
# 1. Проверка доступности двух GPU Tesla T4
import torch
import os

print("--- Проверка NVIDIA GPU ---")
os.system("nvidia-smi")

print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"Device count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    name = torch.cuda.get_device_name(i)
    mem = torch.cuda.get_device_properties(i).total_memory / (1024**3)
    print(f"  GPU {i}: {name} ({mem:.2f} GB VRAM)")


In [ ]:
# 2. Установка современного стека библиотек
!pip install -q -U transformers accelerate bitsandbytes gradio


In [ ]:
# 3. Загрузка Qwen 2.5 7B Instruct с балансировкой по двум GPU (device_map='auto')
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
print(f"Загрузка модели {MODEL_ID}...")

# 4-битное квантование NF4 для максимальной скорости и экономии VRAM
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)

print("Модель успешно загружена на оба GPU!")
print("Распределение слоев по устройствам:", model.hf_device_map)


In [ ]:
# 4. Тестовый запрос к модели
prompt = "Привет! Напиши краткое приветствие и подтверди, что работаешь на сервере с двумя GPU T4."
messages = [{"role": "user", "content": prompt}]
inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(inputs, max_new_tokens=150, temperature=0.7)

response = tokenizer.decode(outputs[0][len(inputs[0]):], skip_special_tokens=True)
print("\nОтвет модели:\n", response)


In [ ]:
# 5. Запуск веб-чата с публичной ссылкой
import gradio as gr

def chat_fn(message, history):
    messages = []
    for h in history:
        messages.append({"role": "user", "content": h[0]})
        messages.append({"role": "assistant", "content": h[1]})
    messages.append({"role": "user", "content": message})

    prompt_inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model.generate(prompt_inputs, max_new_tokens=256, temperature=0.7)
    answer = tokenizer.decode(out[0][len(prompt_inputs[0]):], skip_special_tokens=True)
    return answer

demo = gr.ChatInterface(
    fn=chat_fn,
    title=f"Kaggle Dual T4 AI Assistant ({MODEL_ID})",
    description="Интерактивный чат на 2x NVIDIA Tesla T4 (Kaggle Cloud)"
)

demo.launch(share=True, debug=False)
